# Week 5 Day 3


In [1]:
from typing import TypedDict

class ResearchState(TypedDict):
    query: str
    plan: str
    retrieved_data: dict
    draft: str
    critique: str
    quality_score: float
    revision_count: int
    max_revisions: int
    final_answer: str
    approved: bool

```mermaid
flowchart TD
    START([START]) --> plan
    plan --> retrieve
    retrieve --> generate
    generate --> format
    format --> END1([END])
```

In [2]:
import json

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

MODEL = "llama-3.3-70b-versatile"
llm = ChatGroq(model=MODEL, temperature=0)

QUALITY_THRESHOLD = 0.8


## Task 2: Build a Linear Graph

Four nodes: plan -> retrieve -> generate -> format. Each node prints the state
after it runs so we can verify updates are flowing correctly before adding
loops or interrupts.


In [3]:
def plan_node(state: ResearchState) -> dict:
    query = state["query"]
    response = llm.invoke(
        f"A client asked: \"{query}\". In one sentence, state a plan "
        f"for how to research and answer this (which products to look up, "
        f"what to compare)."
    )
    print("[plan_node] plan:", response.content)
    return {"plan": response.content}


def retrieve_node(state: ResearchState) -> dict:
    # Simple deterministic retrieval for this workflow: pull all laptop entries
    # from the catalog so generate_node has real data to compare, rather than
    # letting the model invent prices.
    with open('products.json') as f:
        db = json.load(f)
    laptops = {k: v for k, v in db.items() if v["category"] == "laptop"}
    print("[retrieve_node] retrieved:", laptops)
    return {"retrieved_data": laptops}


def generate_node(state: ResearchState) -> dict:
    query = state["query"]
    plan = state["plan"]
    retrieved_data = state["retrieved_data"]
    prompt = (
        f"Client request: {query}\n"
        f"Plan: {plan}\n"
        f"Product data: {json.dumps(retrieved_data)}\n"
    )
    critique = state.get("critique")
    if critique:
        prompt += f"\nPrevious critique to address: {critique}\n"
    prompt += "\nWrite a short product recommendation for the client based on this data."

    response = llm.invoke(prompt)
    revision_count = state.get("revision_count", 0)
    print(f"[generate_node] draft (revision {revision_count}):", response.content)
    return {"draft": response.content}


def format_node(state: ResearchState) -> dict:
    draft = state["draft"]
    quality_score = state.get("quality_score", 0)
    revision_count = state.get("revision_count", 0)
    formatted = (
        f"RECOMMENDATION\n"
        f"---------------\n"
        f"{draft}\n\n"
        f"(quality score: {quality_score:.2f}, "
        f"revisions used: {revision_count})"
    )
    print("[format_node] final_answer set.")
    return {"final_answer": formatted}

In [4]:
linear_graph_builder = StateGraph(ResearchState)
linear_graph_builder.add_node("plan", plan_node)
linear_graph_builder.add_node("retrieve", retrieve_node)
linear_graph_builder.add_node("generate", generate_node)
linear_graph_builder.add_node("format", format_node)

linear_graph_builder.add_edge(START, "plan")
linear_graph_builder.add_edge("plan", "retrieve")
linear_graph_builder.add_edge("retrieve", "generate")
linear_graph_builder.add_edge("generate", "format")
linear_graph_builder.add_edge("format", END)

linear_graph = linear_graph_builder.compile()

result = linear_graph.invoke({
    "query": "Recommend a laptop for a budget-conscious client.",
    "revision_count": 0,
    "max_revisions": 2,
})

print("\nFINAL STATE:")
print(json.dumps(result, indent=2, default=str))

[plan_node] plan: To research and recommend a laptop for a budget-conscious client, I will look up and compare affordable options from reputable brands such as Acer, Lenovo, and HP, considering factors like processor speed, memory, storage, display quality, and battery life, within a price range of $300-$800, to find the best value for their money.
[retrieve_node] retrieved: {'laptop_a': {'name': 'Laptop A (Budget)', 'price_usd': 550, 'category': 'laptop'}, 'laptop_b': {'name': 'Laptop B (Mid-range)', 'price_usd': 950, 'category': 'laptop'}, 'laptop_c': {'name': 'Laptop C (Premium)', 'price_usd': 1800, 'category': 'laptop'}}
[generate_node] draft (revision 0): Based on your budget-conscious requirements, I recommend considering "Laptop A (Budget)" priced at $550. Although the provided data is limited, Laptop A falls within your desired price range of $300-$800 and is categorized as a budget option, suggesting it may offer a suitable balance of features and affordability. However, to en

## Task 3: Add Conditional Edges & Cycles

Add a critique node between generate and format. A conditional edge routes back to
generate (self-correction loop) if the self-assessed quality score is below
QUALITY_THRESHOLD **and** revision_count < max_revisions; otherwise it proceeds to
format. revision_count in state is the loop-guard.


```mermaid
flowchart TD
    START([START]) --> plan
    plan --> retrieve
    retrieve --> generate
    generate --> critique
    critique -- quality low & revisions < max --> generate
    critique -- quality high or max revisions --> format
    format --> END1([END])
```

In [5]:
def critique_node(state: ResearchState) -> dict:
    draft = state["draft"]
    prompt = (
        f"Critique this product recommendation for accuracy and usefulness to a "
        f"budget-conscious client. Recommendation: {draft}\n\n"
        f"Respond with a JSON object only: "
        f'{{"score": <float 0-1>, "feedback": "<one sentence>"}}'
    )
    response = llm.invoke(prompt)
    try:
        parsed = json.loads(response.content)
        score = float(parsed["score"])
        feedback = parsed["feedback"]
    except Exception:
        # If the model doesn't return clean JSON, fail safe: treat as passable
        # rather than looping forever on a parsing issue.
        score, feedback = 0.85, "Could not parse critique JSON; accepting draft."

    revision_count = state.get("revision_count", 0)
    print(f"[critique_node] score={score:.2f} feedback={feedback!r} revision_count={revision_count}")
    return {"quality_score": score, "critique": feedback}


def route_after_critique(state: ResearchState) -> str:
    below_threshold = state["quality_score"] < QUALITY_THRESHOLD
    revisions_left = state.get("revision_count", 0) < state.get("max_revisions", 2)
    if below_threshold and revisions_left:
        print("[route_after_critique] -> generate (looping back)")
        return "generate"
    print("[route_after_critique] -> format (accepted)")
    return "format"


def increment_revision_node(state: ResearchState) -> dict:
    # Separate tiny node whose only job is to bump the loop counter, kept
    # explicit and visible in the graph rather than hidden inside generate_node.
    return {"revision_count": state.get("revision_count", 0) + 1}

In [6]:
cyclical_graph_builder = StateGraph(ResearchState)
cyclical_graph_builder.add_node("plan", plan_node)
cyclical_graph_builder.add_node("retrieve", retrieve_node)
cyclical_graph_builder.add_node("generate", generate_node)
cyclical_graph_builder.add_node("critique", critique_node)
cyclical_graph_builder.add_node("increment_revision", increment_revision_node)
cyclical_graph_builder.add_node("format", format_node)

cyclical_graph_builder.add_edge(START, "plan")
cyclical_graph_builder.add_edge("plan", "retrieve")
cyclical_graph_builder.add_edge("retrieve", "generate")
cyclical_graph_builder.add_edge("generate", "critique")
cyclical_graph_builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {"generate": "increment_revision", "format": "format"},
)
cyclical_graph_builder.add_edge("increment_revision", "generate")
cyclical_graph_builder.add_edge("format", END)

cyclical_graph = cyclical_graph_builder.compile()

result = cyclical_graph.invoke({
    "query": "Recommend a laptop for a budget-conscious client.",
    "revision_count": 0,
    "max_revisions": 2,
})

print("\nFINAL STATE:")
print(json.dumps(result, indent=2, default=str))

[plan_node] plan: To research and recommend a laptop for a budget-conscious client, I will look up and compare affordable options from reputable brands such as Acer, Lenovo, and HP, considering factors like processor speed, memory, storage, display quality, and battery life, within a price range of $300-$800, to find the best value for their money.
[retrieve_node] retrieved: {'laptop_a': {'name': 'Laptop A (Budget)', 'price_usd': 550, 'category': 'laptop'}, 'laptop_b': {'name': 'Laptop B (Mid-range)', 'price_usd': 950, 'category': 'laptop'}, 'laptop_c': {'name': 'Laptop C (Premium)', 'price_usd': 1800, 'category': 'laptop'}}
[generate_node] draft (revision 0): Based on your budget-conscious requirements, I recommend considering "Laptop A (Budget)" priced at $550. This option falls within your desired price range of $300-$800 and offers a great balance of affordability and functionality. While it may not have all the high-end features of more expensive models like "Laptop B (Mid-range)"

Why this loop-back pattern is hard to express cleanly in a plain AgentExecutor
but natural in LangGraph: AgentExecutor only exposes one implicit loop  call model,
check for a tool call, execute, repeat  with a single flat max_iterations cutoff; there
is no way to say "loop back to this specific step under this specific condition, but
only up to N times, while a different step runs once." LangGraph's conditional edges let
you name that exact transition (critique -> generate when quality is low) as a first-class
part of the graph, with its own counter in state, while every other edge stays linear 
you'd have to hand-roll custom branching logic inside AgentExecutor's single loop to get
the same effect, fighting against its architecture rather than using it.


## Task 4: Human-in-the-Loop & Interrupts

Add a human_approval node that calls interrupt() before the "risky" action 
here, sending a purchase-confirmation email. The graph pauses, is resumed with
Command(resume=...) after simulated human approval/rejection, and routes accordingly.
A checkpointer is required for interrupt() to work at all (LangGraph raises at compile
time otherwise)  see Task 5 for why.


```mermaid
flowchart TD
    START([START]) --> plan
    plan --> retrieve
    retrieve --> generate
    generate --> critique
    critique -- quality low & revisions < max --> generate
    critique -- quality high or max revisions --> format
    format --> human_approval
    human_approval -- approved --> send_email --> END1([END])
    human_approval -- rejected --> END2([END - cancelled])
```

In [7]:
def human_approval_node(state: ResearchState) -> dict:
    decision = interrupt({
        "question": "Approve sending this recommendation as a purchase-confirmation email to the client?",
        "preview": state["final_answer"],
    })
    print(f"[human_approval_node] resumed with decision: {decision}")
    return {"approved": decision == "yes"}


def send_email_node(state: ResearchState) -> dict:
    # Simulated "risky" external action -- in a real product this would call
    # an email API. We only ever reach this node if a human approved it.
    print("[send_email_node] SENDING EMAIL (simulated):")
    print(state["final_answer"])
    return {}


def route_after_approval(state: ResearchState) -> str:
    return "send_email" if state.get("approved") else END

In [8]:
checkpointer = MemorySaver()

hitl_graph_builder = StateGraph(ResearchState)
hitl_graph_builder.add_node("plan", plan_node)
hitl_graph_builder.add_node("retrieve", retrieve_node)
hitl_graph_builder.add_node("generate", generate_node)
hitl_graph_builder.add_node("critique", critique_node)
hitl_graph_builder.add_node("increment_revision", increment_revision_node)
hitl_graph_builder.add_node("format", format_node)
hitl_graph_builder.add_node("human_approval", human_approval_node)
hitl_graph_builder.add_node("send_email", send_email_node)

hitl_graph_builder.add_edge(START, "plan")
hitl_graph_builder.add_edge("plan", "retrieve")
hitl_graph_builder.add_edge("retrieve", "generate")
hitl_graph_builder.add_edge("generate", "critique")
hitl_graph_builder.add_conditional_edges(
    "critique",
    route_after_critique,
    {"generate": "increment_revision", "format": "format"},
)
hitl_graph_builder.add_edge("increment_revision", "generate")
hitl_graph_builder.add_edge("format", "human_approval")
hitl_graph_builder.add_conditional_edges(
    "human_approval",
    route_after_approval,
    {"send_email": "send_email", END: END},
)
hitl_graph_builder.add_edge("send_email", END)

hitl_graph = hitl_graph_builder.compile(checkpointer=checkpointer)

thread_config = {"configurable": {"thread_id": "client-approval-1"}}

for event in hitl_graph.stream(
    {
        "query": "Recommend a laptop for a budget-conscious client.",
        "revision_count": 0,
        "max_revisions": 2,
    },
    config=thread_config,
    stream_mode="updates",
):
    print("EVENT:", list(event.keys()))

[plan_node] plan: To research and recommend a laptop for a budget-conscious client, I will look up and compare affordable options from reputable brands such as Acer, Lenovo, and HP, considering factors like processor speed, memory, storage, display quality, and battery life, within a price range of $300-$800, to find the best value for their money.
EVENT: ['plan']
[retrieve_node] retrieved: {'laptop_a': {'name': 'Laptop A (Budget)', 'price_usd': 550, 'category': 'laptop'}, 'laptop_b': {'name': 'Laptop B (Mid-range)', 'price_usd': 950, 'category': 'laptop'}, 'laptop_c': {'name': 'Laptop C (Premium)', 'price_usd': 1800, 'category': 'laptop'}}
EVENT: ['retrieve']
[generate_node] draft (revision 0): Based on your budget-conscious requirements, I recommend considering "Laptop A (Budget)" priced at $550. Although the provided data is limited, Laptop A falls within your desired price range of $300-$800 and is categorized as a budget option. While it may not have the most advanced features, it

The stream above stops at `human_approval` — the graph is now paused. Resume it with `Command(resume=...)`:

In [9]:
# Simulate a human APPROVING the action
for event in hitl_graph.stream(
    Command(resume="yes"),
    config=thread_config,
    stream_mode="updates",
):
    print("EVENT:", list(event.keys()))

final_state = hitl_graph.get_state(thread_config)
print("\nFinal values after approval:")
print(json.dumps(final_state.values, indent=2, default=str))

[human_approval_node] resumed with decision: yes
EVENT: ['human_approval']
[send_email_node] SENDING EMAIL (simulated):
RECOMMENDATION
---------------
Based on your budget-conscious requirements, I recommend considering "Laptop A (Budget)" priced at $550. Although the provided data is limited, Laptop A falls within your desired price range of $300-$800 and is categorized as a budget option. While it may not have the most advanced features, it should provide a good balance of performance and affordability. I would suggest comparing its specifications, such as processor speed, memory, and storage, to ensure it meets your specific needs. Additionally, I can look into other options from brands like Acer, Lenovo, and HP to find the best value for your money. Would you like me to research more options or provide more details on Laptop A?

(quality score: 0.80, revisions used: 0)
EVENT: ['send_email']

Final values after approval:
{
  "query": "Recommend a laptop for a budget-conscious client

In [10]:
# Now demonstrate the REJECTED path on a fresh thread
reject_thread_config = {"configurable": {"thread_id": "client-approval-2"}}

for event in hitl_graph.stream(
    {
        "query": "Recommend a laptop for a budget-conscious client.",
        "revision_count": 0,
        "max_revisions": 2,
    },
    config=reject_thread_config,
    stream_mode="updates",
):
    print("EVENT:", list(event.keys()))

# Simulate a human REJECTING the action
for event in hitl_graph.stream(
    Command(resume="no"),
    config=reject_thread_config,
    stream_mode="updates",
):
    print("EVENT:", list(event.keys()))

print("\nRejected-thread final state approved flag:",
      hitl_graph.get_state(reject_thread_config).values.get("approved"))

[plan_node] plan: To research and recommend a laptop for a budget-conscious client, I will look up and compare affordable options from reputable brands such as Acer, Lenovo, and HP, considering factors like processor speed, memory, storage, display quality, and battery life, within a price range of $300-$800, to find the best value for their money.
EVENT: ['plan']
[retrieve_node] retrieved: {'laptop_a': {'name': 'Laptop A (Budget)', 'price_usd': 550, 'category': 'laptop'}, 'laptop_b': {'name': 'Laptop B (Mid-range)', 'price_usd': 950, 'category': 'laptop'}, 'laptop_c': {'name': 'Laptop C (Premium)', 'price_usd': 1800, 'category': 'laptop'}}
EVENT: ['retrieve']
[generate_node] draft (revision 0): Based on your budget-conscious requirements, I recommend considering "Laptop A (Budget)" priced at $550. Although the provided data is limited, Laptop A falls within your desired price range of $300-$800 and is categorized as a budget option. While it may not have the most advanced features, it

**When should a real product require human-in-the-loop, and when is full autonomy
acceptable?** Require a human checkpoint whenever an action is hard to reverse, costs
real money, or affects someone outside the system's control  sending an email to a
client, executing a trade, deleting data, or anything with legal/compliance weight. Full
autonomy is reasonable when actions are cheap, reversible, and low-stakes  looking up
data, drafting (but not sending) content, or internal read-only queries, where a mistake
costs a retry rather than a real-world consequence.


## Task 5: Persistence & Debugging

The MemorySaver checkpointer from Task 4 already persists state per thread_id across
separate .stream()/.invoke() calls  that's what let us resume the paused approval
graph in its own Python call after the interrupt. Below: an explicit demonstration of
resuming a session started earlier, plus using get_state_history() for time-travel
debugging.


In [11]:
# Demonstrate resuming a session across what simulates a separate run: we
# reuse the SAME thread_id as the approved run above and start a brand new
# query, and confirm the earlier state is still retrievable independently.
resumed_state = hitl_graph.get_state(thread_config)
print("Resumed state still accessible from earlier session:")
print("revision_count:", resumed_state.values.get("revision_count"))
print("approved:", resumed_state.values.get("approved"))
print("final_answer present:", "final_answer" in resumed_state.values)

Resumed state still accessible from earlier session:
revision_count: 0
approved: True
final_answer present: True


In [12]:
# Time-travel: walk the full state history of the approved run to see every
# checkpoint the graph passed through, in order.
print("STATE HISTORY (most recent first):\n")
for i, snapshot in enumerate(hitl_graph.get_state_history(thread_config)):
    next_node = snapshot.next[0] if snapshot.next else "(end)"
    print(f"[{i}] next_node={next_node!r} revision_count={snapshot.values.get('revision_count')} "
          f"quality_score={snapshot.values.get('quality_score')}")

STATE HISTORY (most recent first):

[0] next_node='(end)' revision_count=0 quality_score=0.8
[1] next_node='send_email' revision_count=0 quality_score=0.8
[2] next_node='human_approval' revision_count=0 quality_score=0.8
[3] next_node='format' revision_count=0 quality_score=0.8
[4] next_node='critique' revision_count=0 quality_score=None
[5] next_node='generate' revision_count=0 quality_score=None
[6] next_node='retrieve' revision_count=0 quality_score=None
[7] next_node='plan' revision_count=0 quality_score=None
[8] next_node='__start__' revision_count=None quality_score=None


To actually replay/branch from an earlier point (not just inspect it), you'd take
one of the snapshot.config objects from the history above and call
hitl_graph.invoke(new_input_or_None, config=snapshot.config)  LangGraph resumes
execution from that exact checkpoint instead of the latest one, which is the basis of
"time travel" debugging: you can replay a run from any past node, optionally with a
tweaked state, to see how a different decision downstream would have played out.

### LangChain AgentExecutor vs. LangGraph  when to reach for each

AgentExecutor is the right choice for a genuinely single-loop agent: one system
prompt, one set of tools, no need to branch, no need to pause for a human, and where
"call model, maybe call a tool, repeat until done" fully describes the task. It's less
code, faster to stand up, and the abstraction fits.

LangGraph is the right choice as soon as the workflow has real structure: multiple
distinct stages (plan/retrieve/generate/critique/format), a self-correction loop with a
retry cap, human approval gates, or a need to persist/resume/replay state across sessions
 all things AgentExecutor either can't express cleanly or doesn't support at all
(interrupts, per-thread persistence, time travel). The trade-off is more upfront design
(you have to define the state schema and graph shape explicitly) in exchange for
explicit, debuggable control flow instead of one implicit loop.
